In [1]:
import cugraph

In [2]:
import networkit

In [5]:
!python -m ensurepip --upgrade

Looking in links: /tmp/tmpvck_r3c9
Processing /tmp/tmpvck_r3c9/pip-24.0-py3-none-any.whl


In [6]:
!python -m pip install --upgrade pip

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0


In [7]:
!python -m pip --version

pip 26.0.1 from /home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/pip (python 3.12)


In [12]:
import sys
!{sys.executable} -m pip install polars

  Using cached polars-1.38.1-py3-none-any.whl.metadata (10 kB)
  Using cached polars_runtime_32-1.38.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.5 kB)
Using cached polars-1.38.1-py3-none-any.whl (810 kB)
Using cached polars_runtime_32-1.38.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (45.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]


In [11]:
import sys
print(sys.executable)

/home/tracerofjageum/my-lab/.venv/bin/python3


In [8]:
!python -m pip install pandas numpy scikit-learn xgboost catboost lightgbm networkx matplotlib seaborn imbalanced-learn shap

  Using cached xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl.metadata (2.1 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl (131.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 32.1 MB/s  0:00:03m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 47.6 MB/s  0:00:00
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.0 MB/s  0:00:00
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 71.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [imbalanced-learn][imbalanced-learn]


In [2]:
import os
import polars as pl         # Polars의 고속 연산 능력을 활용하여 대규모 시계열 데이터를 효율적으로 처리하는 코드를 구성했습니다.
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb

import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# cuGraph 및 NetworKit 임포트
try:
    import cugraph
    import cudf
    CUGRAPH_AVAILABLE = True
    print("✅ cuGraph 사용 가능")
except ImportError:
    CUGRAPH_AVAILABLE = False
    print("⚠️  cuGraph 없음 - NetworkX 대체 사용")
    import networkx as nx

try:
    import networkit as nk
    NETWORKIT_AVAILABLE = True
    print("✅ NetworKit 사용 가능")
except ImportError:
    NETWORKIT_AVAILABLE = False
    print("⚠️  NetworKit 없음 - NetworkX 대체 사용")
    import networkx as nx

✅ cuGraph 사용 가능


✅ NetworKit 사용 가능


In [6]:
import os
print(os.getcwd())

/home/tracerofjageum/my-lab


In [11]:
import os

print(os.listdir("/home/tracerofjageum/my-lab/첫번째 모델"))

['first_model (1).ipynb', '.ipynb_checkpoints', 'aml_features_final_advanced (1).parquet', 'first_baselinemodel_feature (2).ipynb', 'HI-Medium_Master.parquet']


In [8]:
import os
print("현재 위치:", os.getcwd())
print("파일 목록:", os.listdir())


현재 위치: /home/tracerofjageum/my-lab
파일 목록: ['Untitled1.ipynb', '.venv', 'Untitled.ipynb', '.ipynb_checkpoints', '첫번째 모델']


In [15]:
# 1. 아주 일부분(10행)만 읽어서 컬럼명 확인
print("--- Transaction 파일 컬럼명 ---")
temp_trans = pl.read_csv("/home/tracerofjageum/my-lab/첫번째 모델/HI-Medium_Trans.csv", n_rows=10)
print(temp_trans.columns)

print("\n--- Accounts 파일 컬럼명 ---")
temp_acc = pl.read_csv("/home/tracerofjageum/my-lab/첫번째 모델/HI-Medium_accounts.csv", n_rows=10)
print(temp_acc.columns)

# 2. 경로 설정
base_path = "/home/tracerofjageum/my-lab/첫번째 모델"
trans_file = os.path.join(base_path, "HI-Medium_Trans.csv")
acc_file = os.path.join(base_path, "HI-Medium_accounts.csv")
output_file = os.path.join(base_path, "HI-Medium_Master.parquet")

# 2. [수정] 표준 스키마: Is Laundering을 Int8로 변경
TRANS_OVERRIDES = {
    "Timestamp": pl.Utf8,
    "Account": pl.Categorical,              # Sender Account
    "Account_duplicated_0": pl.Categorical, # Receiver Account
    "Amount Paid": pl.Float32,
    "Is Laundering": pl.Int8                # pl.Boolean에서 pl.Int8로 수정
}
ACC_OVERRIDES = {
    "Account Number": pl.Categorical,
    "Bank Name": pl.Categorical,
    "Entity Name": pl.Categorical
}

print("--- [1] 데이터 로드 및 병합 시작 ---")
# schema_overrides를 사용하여 선언한 타입만 강제하고 나머지는 자동 추론합니다.
q_trans = pl.scan_csv(trans_file, schema_overrides=TRANS_OVERRIDES)
q_acc = pl.scan_csv(acc_file, schema_overrides=ACC_OVERRIDES)

# 3. 마스터 테이블 로직
q_master = (
    q_trans
    # (1) 송신자(Sender) 정보 결합
    .join(q_acc, left_on="Account", right_on="Account Number", how="left")
    .rename({"Bank Name": "src_bank", "Entity Name": "src_entity", "Entity ID": "src_entity_id"})

    # (2) 수신자(Receiver) 정보 결합
    .join(q_acc, left_on="Account_duplicated_0", right_on="Account Number", how="left")
    .rename({"Bank Name": "dst_bank", "Entity Name": "dst_entity", "Entity ID": "dst_entity_id"})

    # (3) 컬럼명 정리 및 데이터 타입 최종 보정
    .rename({"Account": "from_acc", "Account_duplicated_0": "to_acc"})
    .with_columns([
        pl.col("Is Laundering").cast(pl.Boolean) # 여기서 안전하게 bool로 변환
    ])
)

print("--- [2] 마스터 Parquet 파일 굽는 중... (Streaming) ---")
# sink_parquet로 메모리 효율 극대화
q_master.sink_parquet(output_file)

print(f"✅ 성공! 마스터 파일 생성 완료: {output_file}")

--- Transaction 파일 컬럼명 ---
['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account_duplicated_0', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

--- Accounts 파일 컬럼명 ---
['Bank Name', 'Bank ID', 'Account Number', 'Entity ID', 'Entity Name']
--- [1] 데이터 로드 및 병합 시작 ---
--- [2] 마스터 Parquet 파일 굽는 중... (Streaming) ---
✅ 성공! 마스터 파일 생성 완료: /home/tracerofjageum/my-lab/첫번째 모델/HI-Medium_Master.parquet


In [16]:
import polars as pl

# 1. 딱 한 줄만 '실제로' 가져와서 확인 (item()은 단일 값을 꺼낼 때 씁니다)
sample_trans = q_trans.select("Account").head(1).collect().item()
sample_acc = q_acc.select("Account Number").head(1).collect().item()

print(f"--- [데이터 타입 & 값 검증] ---")
print(f"Trans 계좌 샘플:   '{sample_trans}' (타입: {type(sample_trans)}, 길이: {len(str(sample_trans))})")
print(f"Accounts 계좌 샘플: '{sample_acc}' (타입: {type(sample_acc)}, 길이: {len(str(sample_acc))})")

# 2. [심화] 공백이나 특수문자가 숨어있는지 더 정밀하게 확인
if str(sample_trans) == str(sample_acc):
    print("\n✅ 일단 샘플상으로는 두 값이 완벽히 일치합니다!")
else:
    print("\n⚠️ 경고: 눈으로 보기엔 같아도 내부 값이 다를 수 있습니다.")
    print(f"Trans 값 repr: {repr(sample_trans)}")
    print(f"Acc   값 repr: {repr(sample_acc)}")


--- [데이터 타입 & 값 검증] ---
Trans 계좌 샘플:   '800104D70' (타입: <class 'str'>, 길이: 9)
Accounts 계좌 샘플: '817D00980' (타입: <class 'str'>, 길이: 9)

⚠️ 경고: 눈으로 보기엔 같아도 내부 값이 다를 수 있습니다.
Trans 값 repr: '800104D70'
Acc   값 repr: '817D00980'


In [17]:
import polars as pl

# 1. 각 파일의 고유 계좌 번호 추출 (Lazy)
unique_trans_acc = q_trans.select(pl.col("Account")).unique()
unique_master_acc = q_acc.select(pl.col("Account Number")).unique()

# 2. 거래 데이터에 있는 계좌 중, 마스터 정보에 없는 계좌 찾기
# (Left Anti Join은 '왼쪽에는 있지만 오른쪽에는 없는' 데이터만 골라냅니다)
missing_accounts = unique_trans_acc.join(
    unique_master_acc,
    left_on="Account",
    right_on="Account Number",
    how="anti"
).collect()

# 3. 매칭 실패율 계산
total_unique_trans = unique_trans_acc.collect().height
missing_count = missing_accounts.height
fail_rate = (missing_count / total_unique_trans) * 100

print(f"--- [매칭 무결성 검증 결과] ---")
print(f"거래 내 고유 계좌 수: {total_unique_trans:,}개")
print(f"마스터 정보에 없는 계좌 수: {missing_count:,}개")
print(f"매칭 실패율: {fail_rate:.4f}%")

if fail_rate < 1.0:
    print("\n✅ 결과: 매칭률이 매우 높습니다! 안심하고 병합하셔도 됩니다.")
else:
    print("\n⚠️ 경고: 매칭 실패율이 높습니다. 계좌 번호의 공백이나 형식을 다시 확인해야 합니다.")

--- [매칭 무결성 검증 결과] ---
거래 내 고유 계좌 수: 1,812,747개
마스터 정보에 없는 계좌 수: 0개
매칭 실패율: 0.0000%

✅ 결과: 매칭률이 매우 높습니다! 안심하고 병합하셔도 됩니다.


In [19]:
import polars as pl

# 1. 방금 만든 따끈따끈한 마스터 파일 읽기 (딱 5줄만)
final_check = pl.read_parquet('/home/tracerofjageum/my-lab/첫번째 모델/HI-Medium_Master.parquet', n_rows=5)

# 2. 송신자/수신자 은행 이름이 잘 들어있나 확인
print(final_check.select(["from_acc", "src_bank", "to_acc", "dst_bank", "Is Laundering"]))

# 3. 만약 Null이 없다면 통과!
print("\n--- [최종 결과] ---")
print("✅ 데이터가 빈칸 없이 꽉 차있습니다. 이제 팀원들에게 배포해도 좋습니다!")

shape: (5, 5)
┌───────────┬─────────────────────────────┬───────────┬────────────────────────────┬───────────────┐
│ from_acc  ┆ src_bank                    ┆ to_acc    ┆ dst_bank                   ┆ Is Laundering │
│ ---       ┆ ---                         ┆ ---       ┆ ---                        ┆ ---           │
│ cat       ┆ cat                         ┆ cat       ┆ cat                        ┆ bool          │
╞═══════════╪═════════════════════════════╪═══════════╪════════════════════════════╪═══════════════╡
│ 800BAF590 ┆ Savings Bank of Dallas      ┆ 800BAF590 ┆ Savings Bank of Dallas     ┆ false         │
│ 800193A70 ┆ Savings Bank of Augusta     ┆ 800193A70 ┆ Savings Bank of Augusta    ┆ false         │
│ 804389A70 ┆ Bank of Albany              ┆ 828023980 ┆ First Bank of Houston      ┆ false         │
│ 80145B970 ┆ National Bank of Newport    ┆ 80145B970 ┆ National Bank of Newport   ┆ false         │
│ 800119E40 ┆ National Bank of Los        ┆ 800119E40 ┆ National Bank of Los 

In [21]:
final_check.select(pl.all()).head(5)

Timestamp,From Bank,from_acc,To Bank,to_acc,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,src_bank,Bank ID,src_entity_id,src_entity,dst_bank,Bank ID_right,dst_entity_id,dst_entity
str,i64,cat,i64,cat,f64,str,f32,str,str,bool,cat,i64,str,cat,cat,i64,str,cat
"""2022/09/01 00:24""",32041,"""800BAF590""",32041,"""800BAF590""",5914.14,"""US Dollar""",5914.140137,"""US Dollar""","""Reinvestment""",false,"""Savings Bank of Dallas""",32041,"""2AA20608620""","""Partnership #20""","""Savings Bank of Dallas""",32041,"""2AA20608620""","""Partnership #20"""
"""2022/09/01 00:12""",3305,"""800193A70""",3305,"""800193A70""",951.94,"""US Dollar""",951.940002,"""US Dollar""","""Reinvestment""",false,"""Savings Bank of Augusta""",3305,"""2AA207BA540""","""Corporation #155""","""Savings Bank of Augusta""",3305,"""2AA207BA540""","""Corporation #155"""
"""2022/09/01 00:24""",28291,"""804389A70""",198133,"""828023980""",11.67,"""US Dollar""",11.67,"""US Dollar""","""Credit Card""",false,"""Bank of Albany""",28291,"""2AA205B6200""","""Sole Proprietorship #18858""","""First Bank of Houston""",198133,"""2AA1DD0E190""","""Corporation #88"""
"""2022/09/01 00:05""",33548,"""80145B970""",33548,"""80145B970""",2081.03,"""US Dollar""",2081.030029,"""US Dollar""","""Reinvestment""",false,"""National Bank of Newport""",33548,"""2AA200D8ED0""","""Partnership #26""","""National Bank of Newport""",33548,"""2AA200D8ED0""","""Partnership #26"""
"""2022/09/01 00:10""",20,"""800119E40""",20,"""800119E40""",9.74,"""US Dollar""",9.74,"""US Dollar""","""Reinvestment""",false,"""National Bank of Los Angeles""",20,"""2AA208154E0""","""Corporation #124""","""National Bank of Los Angeles""",20,"""2AA208154E0""","""Corporation #124"""


In [44]:
print("=" * 80)
print("고성능 그래프 피처 생성 시작 (cuGraph + NetworKit)")
print("=" * 80)

# # 컬럼명 표준화
# all_cols = final_check.collect_schema().names()
# amount_col = [c for c in all_cols if "Amount" in c][0]
# final_check = final_check.rename({amount_col: "Amount"})


# 사기 계좌 사전 추출
fraud_accounts = (
    final_check
    .filter(pl.col("Is Laundering") == 1)
    .select("from_acc")
    .unique()
    .to_series()
)

fraud_set = set(fraud_accounts.to_list())

# 기본 피처 및 시간 배치 설정
final_check = final_check.with_columns([
    pl.col("from_acc").is_in(fraud_accounts).cast(pl.Int8).alias("is_fraud_node"),
    pl.col("Timestamp").dt.truncate("1h").alias("time_batch")
])

# 전체 데이터 수집
df = final_check
print(f"✅ 데이터 로드 완료: {df.shape[0]:,} 거래, {df.shape[1]} 컬럼")

# Pandas로 변환
df_pd = df.to_pandas()

# 2. 노드 ID 매핑 생성 (문자열 → 정수)
print("\n" + "=" * 80)
print("노드 ID 매핑 생성 중...")
print("=" * 80)

all_accounts = pd.concat([df_pd['from_acc'], df_pd['to_acc']]).unique()
node_to_id = {node: idx for idx, node in enumerate(all_accounts)}
id_to_node = {idx: node for node, idx in node_to_id.items()}

print(f"✅ 고유 노드 수: {len(all_accounts):,}")

# 엣지 리스트 생성 (정수 ID)
edges_df = df_pd[['from_acc', 'to_acc', 'Amount Paid', 'Is Laundering']].copy()
edges_df['src'] = edges_df['from_acc'].map(node_to_id)
edges_df['dst'] = edges_df['to_acc'].map(node_to_id)
edges_df = edges_df[['src', 'dst', 'Amount Paid', 'Is Laundering']]

edges_df = edges_df.rename(columns={'Amount Paid': 'Amount'})   #왜 추가됨?

# 엣지 집계 (동일 엣지 합치기)
edge_agg = edges_df.groupby(['src', 'dst']).agg({
    'Amount': 'sum',
    'Is Laundering': 'max'
}).reset_index()

print(f"✅ 엣지 수: {len(edge_agg):,}")

# =============================================================================
# 3. cuGraph를 사용한 고속 그래프 피처 계산
# =============================================================================
if CUGRAPH_AVAILABLE:
    print("\n" + "=" * 80)
    print("cuGraph로 그래프 피처 계산 중...")
    print("=" * 80)
    
    # cuDF DataFrame으로 변환
    cu_edges = cudf.DataFrame({
        'src': edge_agg['src'].astype('int32').values,
        'dst': edge_agg['dst'].astype('int32').values,
        'weight': edge_agg['Amount'].astype('float32').values
    })
    cu_edges = cu_edges.rename(columns={"Amount": "weight"})
    
    # Directed Graph (centrality용)
    G_directed = cugraph.Graph(directed=True)
    G_directed.from_cudf_edgelist(
        cu_edges,
        source="src",
        destination="dst",
        edge_attr="weight",
        renumber=False
    )

    # Undirected Graph (community / clustering용)
    G_undirected = cugraph.Graph(directed=False)
    G_undirected.from_cudf_edgelist(
        cu_edges,
        source="src",
        destination="dst",
        edge_attr="weight",
        renumber=False
    )
    
    print(f"✅ cuGraph 그래프 생성 완료")
    
    # 3-1. PageRank (GPU 가속)
    print("📊 [cuGraph] PageRank 계산 중...")
    pagerank_df = cugraph.pagerank(G_directed)
    pagerank_dict = dict(zip(pagerank_df['vertex'].to_pandas(), pagerank_df['pagerank'].to_pandas()))
    print(f"✅ PageRank 완료")
    
    # 3-2. Degree (GPU 가속)
    print("📊 [cuGraph] Degree 계산 중...")
    in_degree_df = G_directed.in_degree()
    out_degree_df = G_directed.out_degree()
    
    in_degree_dict = dict(zip(in_degree_df['vertex'].to_pandas(), in_degree_df['degree'].to_pandas()))
    out_degree_dict = dict(zip(out_degree_df['vertex'].to_pandas(), out_degree_df['degree'].to_pandas()))
    print(f"✅ Degree 완료")
    
    # 3-3. Louvain 커뮤니티 탐지 (GPU 가속)
    print("📊 [cuGraph] Louvain 커뮤니티 탐지 중...")
    community_df, modularity_score = cugraph.louvain(G_undirected)

    community_dict = dict(
        zip(
            community_df['vertex'].to_pandas(),
            community_df['partition'].to_pandas()
        )
    )
    print(f"✅ 커뮤니티 탐지 완료: {community_df['partition'].nunique()} 개 커뮤니티")
    print(f"✅ Modularity: {modularity_score}")
    
    # 3-4. Degree Centrality 계산
    print("📊 [cuGraph] Degree Centrality 계산 중...")
    num_nodes = len(all_accounts)
    degree_centrality_dict = {node: out_degree_dict.get(node, 0) / (num_nodes - 1) for node in range(num_nodes)}
    in_degree_centrality_dict = {node: in_degree_dict.get(node, 0) / (num_nodes - 1) for node in range(num_nodes)}
    out_degree_centrality_dict = {node: out_degree_dict.get(node, 0) / (num_nodes - 1) for node in range(num_nodes)}
    print(f"✅ Degree Centrality 완료")
    
    # 3-5. Clustering Coefficient (GPU 가속)
    print("📊 [cuGraph] Clustering Coefficient 계산 중...")
    # cuGraph는 방향 그래프의 clustering을 지원하지 않으므로 무향 그래프로 변환
    clustering_df = cugraph.triangle_count(G_undirected)
    
    # Clustering coefficient 계산: C = 2 * triangles / (degree * (degree - 1))
    clustering_dict = {}
    for _, row in clustering_df.to_pandas().iterrows():
        vertex = int(row['vertex'])
        triangles = int(row['counts'])
        degree = out_degree_dict.get(vertex, 0) + in_degree_dict.get(vertex, 0)  # total degree
        
        if degree >= 2:
            clustering_dict[vertex] = (2 * triangles) / (degree * (degree - 1))
        else:
            clustering_dict[vertex] = 0.0
    
    print(f"✅ Clustering Coefficient 완료")

else:
    # NetworkX fallback
    print("\n⚠️  cuGraph 없음 - NetworkX 사용 (느림)")
    print("=" * 80)
    print("NetworkX로 그래프 구축 중...")
    print("=" * 80)
    
    G_nx = nx.DiGraph()
    for _, row in edge_agg.iterrows():
        G_nx.add_edge(row['src'], row['dst'], weight=row['Amount'])
    
    print("📊 [NetworkX] PageRank 계산 중...")
    pagerank_dict = nx.pagerank(G_nx, alpha=0.85, max_iter=100)
    
    print("📊 [NetworkX] Degree 계산 중...")
    in_degree_dict = dict(G_nx.in_degree())
    out_degree_dict = dict(G_nx.out_degree())
    
    print("📊 [NetworkX] Degree Centrality 계산 중...")
    degree_centrality_dict = nx.degree_centrality(G_nx)
    in_degree_centrality_dict = nx.in_degree_centrality(G_nx)
    out_degree_centrality_dict = nx.out_degree_centrality(G_nx)
    
    print("📊 [NetworkX] 커뮤니티 탐지 중...")
    G_undirected = G_nx.to_undirected()
    communities = nx.community.louvain_communities(G_undirected, seed=42)
    community_dict = {}
    for idx, community in enumerate(communities):
        for node in community:
            community_dict[node] = idx
    
    print("📊 [NetworkX] Clustering Coefficient 계산 중...")
    clustering_dict = nx.clustering(G_undirected)

# =============================================================================
# 4. 후보 노드 필터링 (상위 5~10%)
# =============================================================================
print("\n" + "=" * 80)
print("후보 노드 필터링 (Betweenness 계산용)")
print("=" * 80)

# PageRank + Degree 기준으로 상위 노드 선택
pagerank_values = np.array([pagerank_dict.get(i, 0) for i in range(len(all_accounts))])
total_degree_values = np.array([in_degree_dict.get(i, 0) + out_degree_dict.get(i, 0) for i in range(len(all_accounts))])

# 정규화
pagerank_norm = (pagerank_values - pagerank_values.min()) / (pagerank_values.max() - pagerank_values.min() + 1e-10)
degree_norm = (total_degree_values - total_degree_values.min()) / (total_degree_values.max() - total_degree_values.min() + 1e-10)

# 복합 점수
composite_score = 0.7 * pagerank_norm + 0.3 * degree_norm

# 상위 10% 노드 선택
threshold_percentile = 90
threshold_value = np.percentile(composite_score, threshold_percentile)
important_nodes = np.where(composite_score >= threshold_value)[0]

print(f"✅ 전체 노드: {len(all_accounts):,}")
print(f"✅ 선택된 중요 노드: {len(important_nodes):,} ({len(important_nodes)/len(all_accounts)*100:.1f}%)")

# =============================================================================
# 5. NetworKit을 사용한 Betweenness 및 Local Features 계산
# =============================================================================
if NETWORKIT_AVAILABLE:
    print("\n" + "=" * 80)
    print("NetworKit으로 고급 피처 계산 중...")
    print("=" * 80)
    
    # NetworKit 그래프 생성
    print("📊 [NetworKit] 그래프 생성 중...")
    G_nk = nk.Graph(n=len(all_accounts), weighted=True, directed=False)
    
    for _, row in tqdm(edge_agg.iterrows(), total=len(edge_agg), desc="엣지 추가"):
        G_nk.addEdge(int(row['src']), int(row['dst']), w=float(row['Amount']))
    
    print(f"✅ NetworKit 그래프 생성 완료")
    
    # 5-1. Betweenness Centrality (샘플링 - 중요 노드만)
    print(f"📊 [NetworKit] Betweenness Centrality 계산 중 (상위 {len(important_nodes):,} 노드)...")
    
    # 전체 노드에 대한 근사 Betweenness (매우 빠름)
    approx_betweenness = nk.centrality.ApproxBetweenness(G_nk, epsilon=0.1, delta=0.1)
    approx_betweenness.run()
    betweenness_dict = {i: approx_betweenness.score(i) for i in range(len(all_accounts))}
    
    print(f"✅ Betweenness 완료")

else:
    # NetworkX fallback
    print("\n⚠️  NetworKit 없음 - NetworkX 사용 (느림)")
    print("📊 [NetworkX] Betweenness 계산 중 (샘플링)...")
    betweenness_dict = nx.betweenness_centrality(G_nx, k=min(1000, len(all_accounts)), seed=42)

# =============================================================================
# 6. 그래프 피처 DataFrame 생성
# =============================================================================
print("\n" + "=" * 80)
print("그래프 피처 DataFrame 생성 중...")
print("=" * 80)

graph_features = []

for node_id in tqdm(range(len(all_accounts)), desc="노드 피처 추출"):
    original_account = id_to_node[node_id]
    
    node_features = {
        'account': original_account,
        
        # 사기 여부
        #'is_fraud': 1 if original_account in fraud_accounts.to_list() else 0,    왜 잘못됨????
        'is_fraud': 1 if original_account in fraud_set else 0,
        
        # cuGraph 피처
        'pagerank': pagerank_dict.get(node_id, 0),
        'in_degree': in_degree_dict.get(node_id, 0),
        'out_degree': out_degree_dict.get(node_id, 0),
        'total_degree': in_degree_dict.get(node_id, 0) + out_degree_dict.get(node_id, 0),
        'clustering_coefficient': clustering_dict.get(node_id, 0),
        
        # Degree Centrality
        'degree_centrality': degree_centrality_dict.get(node_id, 0),
        'in_degree_centrality': in_degree_centrality_dict.get(node_id, 0),
        'out_degree_centrality': out_degree_centrality_dict.get(node_id, 0),
        
        # NetworKit 피처
        'betweenness_centrality': betweenness_dict.get(node_id, 0),
        
        # 커뮤니티
        'community_id': community_dict.get(node_id, -1),
    }
    
    graph_features.append(node_features)

graph_features_df = pd.DataFrame(graph_features)

print(f"\n✅ 그래프 피처 추출 완료: {len(graph_features_df):,} 노드")

# =============================================================================
# 6. 그래프 기반 파생 피처
# =============================================================================
print("그래프 기반 파생 피처 생성 중...")

graph_features_df['degree_ratio'] = (
    graph_features_df['in_degree'] /
    (graph_features_df['out_degree'] + 1)
)

graph_features_df['degree_difference'] = (
    graph_features_df['in_degree'] -
    graph_features_df['out_degree']
)

graph_features_df['aml_suspicion_score'] = (
    graph_features_df['pagerank'] *
    graph_features_df['clustering_coefficient'] *
    1000
)

# 커뮤니티 크기
community_sizes = graph_features_df.groupby('community_id').size().to_dict()
graph_features_df['community_size'] = (
    graph_features_df['community_id']
    .map(community_sizes)
    .fillna(0)
    .astype(int)
)

# 커뮤니티 사기 비율
community_fraud_ratio = (
    graph_features_df
    .groupby('community_id')['is_fraud']
    .mean()
    .to_dict()
)

graph_features_df['community_fraud_ratio'] = (
    graph_features_df['community_id']
    .map(community_fraud_ratio)
    .fillna(0)
)

graph_features_df['num_neighbors'] = graph_features_df['out_degree']
graph_features_df['num_predecessors'] = graph_features_df['in_degree']

# =============================================================================
# 7. 저장
# =============================================================================
current_dir = os.getcwd()
output_file = os.path.join(current_dir, "graph_structure_only.parquet")

pl.from_pandas(graph_features_df).write_parquet(output_file)

print("=" * 80)
print("✅ 그래프 구조 피처 생성 완료!")
print("=" * 80)
print(f"📁 저장 파일: {output_file}")
print(f"📊 노드 수: {len(graph_features_df):,}")
print(f"📊 피처 수: {len(graph_features_df.columns)}")

고성능 그래프 피처 생성 시작 (cuGraph + NetworKit)
✅ 데이터 로드 완료: 5 거래, 21 컬럼

노드 ID 매핑 생성 중...
✅ 고유 노드 수: 6
✅ 엣지 수: 25

cuGraph로 그래프 피처 계산 중...
✅ cuGraph 그래프 생성 완료
📊 [cuGraph] PageRank 계산 중...
✅ PageRank 완료
📊 [cuGraph] Degree 계산 중...
✅ Degree 완료
📊 [cuGraph] Louvain 커뮤니티 탐지 중...
✅ 커뮤니티 탐지 완료: 5 개 커뮤니티
✅ Modularity: 0.5013311505317688
📊 [cuGraph] Degree Centrality 계산 중...
✅ Degree Centrality 완료
📊 [cuGraph] Clustering Coefficient 계산 중...
✅ Clustering Coefficient 완료

후보 노드 필터링 (Betweenness 계산용)
✅ 전체 노드: 6
✅ 선택된 중요 노드: 2 (33.3%)

NetworKit으로 고급 피처 계산 중...
📊 [NetworKit] 그래프 생성 중...


엣지 추가: 100%|█████████████████████████████████████████████████████████████| 25/25 [00:00<00:00, 16435.36it/s]


✅ NetworKit 그래프 생성 완료
📊 [NetworKit] Betweenness Centrality 계산 중 (상위 2 노드)...
✅ Betweenness 완료

그래프 피처 DataFrame 생성 중...


노드 피처 추출: 100%|██████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 62137.84it/s]


✅ 그래프 피처 추출 완료: 6 노드
그래프 기반 파생 피처 생성 중...
✅ 그래프 구조 피처 생성 완료!
📁 저장 파일: /home/tracerofjageum/my-lab/graph_structure_only.parquet
📊 노드 수: 6
📊 피처 수: 19


In [45]:
print(os.listdir(base_dir))

['first_model (1).ipynb', 'HI-Medium_accounts.csv', 'HI-Medium_Trans.csv', '.ipynb_checkpoints', 'EDA.ipynb', 'aml_features_final_advanced (1).parquet', 'first_baselinemodel_feature (2).ipynb', 'HI-Medium_Master.parquet']


In [46]:
print(os.path.exists(output_file))
print(os.listdir(current_dir))

True
['Untitled1.ipynb', '.venv', 'graph_structure_only.parquet', '.ipynb_checkpoints', '첫번째 모델', 'second_features.ipynb']


In [48]:
import polars as pl
import os

print("=" * 80)
print("집계 피처 + 그래프 피처 통합 시작")
print("=" * 80)

# 경로 설정
base_dir = "/home/tracerofjageum/my-lab/첫번째 모델/"
aggregation_features_path = os.path.join(base_dir, "aml_features_final_advanced (1).parquet")
graph_features_path = os.path.join(current_dir, "graph_structure_only.parquet")
output_path = os.path.join(base_dir, "aml_features_integrated_final.parquet")

try:
    # =============================================================================
    # 1. 집계 피처 로드
    # =============================================================================
    print("\n📁 집계 피처 로딩 중...")
    df_agg = pl.read_parquet(aggregation_features_path)
    print(f"✅ 집계 피처 Shape: {df_agg.shape}")
    print(f"   - 컬럼 수: {len(df_agg.columns)}")
    print(f"   - 행 수: {len(df_agg):,}")
    
    # =============================================================================
    # 2. 그래프 피처 로드
    # =============================================================================
    print("\n📁 그래프 피처 로딩 중...")
    df_graph = pl.read_parquet(graph_features_path)
    print(f"✅ 그래프 피처 Shape: {df_graph.shape}")
    print(f"   - 컬럼 수: {len(df_graph.columns)}")
    print(f"   - 노드 수: {len(df_graph):,}")
    
    # 그래프 피처 미리보기
    print("\n[그래프 피처 컬럼 목록]")
    print(df_graph.columns)
    
    # =============================================================================
    # 3. 조인 키 정규화
    # =============================================================================
    print("\n🔧 조인 키 정규화 중...")
    
    # 집계 피처: account_id 컬럼이 이미 존재
    # 그래프 피처: account 컬럼을 account_id로 변경
    df_graph = df_graph.rename({"account": "account_id"})
    
    # 타입 통일 (문자열로)
    df_agg = df_agg.with_columns(pl.col("account_id").cast(pl.Utf8))
    df_graph = df_graph.with_columns(pl.col("account_id").cast(pl.Utf8))
    
    print(f"✅ 키 정규화 완료")
    print(f"   - 집계 피처 고유 계좌: {df_agg['account_id'].n_unique():,}")
    print(f"   - 그래프 피처 고유 계좌: {df_graph['account_id'].n_unique():,}")
    
    # =============================================================================
    # 4. 그래프 피처와 조인 (Left Join)
    # =============================================================================
    print("\n🔗 집계 피처 + 그래프 피처 조인 중...")
    
    # is_fraud 컬럼 충돌 방지 (집계 피처에는 is_laundering, 그래프 피처에는 is_fraud)
    # 그래프 피처의 is_fraud는 제거 (집계 피처의 is_laundering을 사용)
    if "is_fraud" in df_graph.columns:
        df_graph = df_graph.drop("is_fraud")
    
    # Left Join: 집계 피처를 기준으로 그래프 피처를 붙임
    df_integrated = df_agg.join(
        df_graph,
        on="account_id",
        how="left"
    )
    
    print(f"✅ 조인 완료!")
    print(f"   - 최종 Shape: {df_integrated.shape}")
    print(f"   - 컬럼 수: {len(df_integrated.columns)}")
    print(f"   - 행 수: {len(df_integrated):,}")
    
    # =============================================================================
    # 5. 결측치 처리 (그래프 피처가 없는 계좌)
    # =============================================================================
    print("\n🧹 그래프 피처 결측치 처리 중...")
    
    # 그래프 피처 컬럼 리스트
    graph_cols = [c for c in df_graph.columns if c != "account_id"]
    
    # 수치형 컬럼: 0으로 채우기
    numeric_graph_cols = [
        "pagerank", "in_degree", "out_degree", "total_degree",
        "clustering_coefficient", "degree_centrality", "in_degree_centrality",
        "out_degree_centrality", "betweenness_centrality",
        "degree_ratio", "degree_difference", "aml_suspicion_score",
        "community_size", "community_fraud_ratio", "num_neighbors", "num_predecessors"
    ]
    
    fill_exprs = []
    for col in numeric_graph_cols:
        if col in df_integrated.columns:
            fill_exprs.append(pl.col(col).fill_null(0))
    
    # 커뮤니티 ID: -1로 채우기
    if "community_id" in df_integrated.columns:
        fill_exprs.append(pl.col("community_id").fill_null(-1))
    
    df_integrated = df_integrated.with_columns(fill_exprs)
    
    print(f"✅ 결측치 처리 완료")
    
    # =============================================================================
    # 6. 추가 파생 피처 (그래프 × 집계)
    # =============================================================================
    print("\n➕ 그래프-집계 교차 피처 생성 중...")
    
    df_integrated = df_integrated.with_columns([
        # PageRank × 거래량 = 중요도 가중 거래량
        (pl.col("pagerank") * pl.col("sum_1h")).alias("pagerank_weighted_volume"),
        
        # 커뮤니티 사기율 × 거래 횟수 = 위험 신호 강도
        (pl.col("community_fraud_ratio") * pl.col("cnt_1h")).alias("community_risk_signal"),
        
        # Betweenness × Burst = 허브 계좌의 폭증 패턴
        (pl.col("betweenness_centrality") * pl.col("burst_ratio")).alias("hub_burst_score"),
        
        # Degree × 위험국가 비율 = 네트워크 위험도
        (pl.col("total_degree") * pl.col("ratio_risk_country")).alias("network_risk_exposure"),
        
        # 클러스터링 계수 × 입출금 비율 = 내부 거래 집중도
        (pl.col("clustering_coefficient") * pl.col("in_out_balance_ratio")).alias("internal_flow_concentration")
    ])
    
    print(f"✅ 교차 피처 5개 생성 완료")
    
    # =============================================================================
    # 7. 최종 저장
    # =============================================================================
    print(f"\n💾 최종 통합 데이터 저장 중: {output_path}")
    
    if os.path.exists(output_path):
        os.remove(output_path)
    
    df_integrated.write_parquet(output_path)
    
    # =============================================================================
    # 8. 결과 요약
    # =============================================================================
    print("\n" + "=" * 80)
    print("✅ 피처 통합 완료!")
    print("=" * 80)
    print(f"📁 저장 경로: {output_path}")
    print(f"📊 최종 Shape: {df_integrated.shape}")
    print(f"   - 총 피처 수: {len(df_integrated.columns)}")
    print(f"   - 총 행 수: {len(df_integrated):,}")
    
    print("\n[피처 카테고리별 개수]")
    
    # 집계 피처
    agg_features = [c for c in df_agg.columns if c not in ["account_id", "time_group"]]
    print(f"   - 집계 피처: {len(agg_features)}개")
    
    # 그래프 피처
    graph_features = [c for c in graph_cols]
    print(f"   - 그래프 피처: {len(graph_features)}개")
    
    # 교차 피처
    cross_features = [
        "pagerank_weighted_volume", "community_risk_signal", 
        "hub_burst_score", "network_risk_exposure", "internal_flow_concentration"
    ]
    print(f"   - 교차 피처: {len(cross_features)}개")
    
    # 타겟 확인
    if "is_laundering" in df_integrated.columns:
        print("\n[타겟 분포]")
        print(df_integrated["is_laundering"].value_counts().sort("is_laundering"))
    
    # 샘플 미리보기
    print("\n[데이터 샘플 (상위 5행)]")
    display_cols = [
        "account_id", "time_group", "sum_1h", "burst_ratio",
        "pagerank", "total_degree", "community_fraud_ratio",
        "hub_burst_score", "is_laundering"
    ]
    display_cols = [c for c in display_cols if c in df_integrated.columns]
    print(df_integrated.select(display_cols).head(5))
    
    print("\n" + "=" * 80)
    print("🎉 모든 작업이 성공적으로 완료되었습니다!")
    print("=" * 80)

except Exception as e:
    print(f"\n❌ 에러 발생: {e}")
    import traceback
    traceback.print_exc()

집계 피처 + 그래프 피처 통합 시작

📁 집계 피처 로딩 중...

❌ 에러 발생: parquet: File out of specification: The file must end with PAR1


Traceback (most recent call last):
  File "/tmp/ipykernel_9097/1695731134.py", line 19, in <module>
    df_agg = pl.read_parquet(aggregation_features_path)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 128, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 128, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/io/parquet/functions.py", line 293, in read_parquet
    return lf.collect()
           ^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 97, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracer

In [1]:
import polars as pl
import os

print("=" * 80)
print("🔗 [피처 통합 프로세스] 집계 피처(V1) + 그래프 피처(V3) 병합 시작")
print("=" * 80)

# -----------------------------------------------------------------------------
# 📂 1. 경로 설정 (여기가 핵심입니다!)
# -----------------------------------------------------------------------------
# [수정됨] 아까 찾은 손상되지 않은 정상 파일의 절대 경로 사용
aggregation_features_path = "/home/tracerofjageum/aml_features_final_advanced.parquet"

# 그래프 피처 파일 (방금 생성한 파일)
# (경로가 헷갈린다면 현재 폴더 기준으로 설정)
graph_features_path = "./graph_structure_only.parquet"

# 최종 결과 저장 경로
output_path = "/home/tracerofjageum/my-lab/첫번째 모델/aml_features_integrated_final.parquet"

try:
    # -------------------------------------------------------------------------
    # 2. 집계 피처 로드 (Base Features)
    # -------------------------------------------------------------------------
    print(f"\n1️⃣ 집계 피처(Base) 로딩 중... \n   📄 경로: {aggregation_features_path}")
    if not os.path.exists(aggregation_features_path):
        raise FileNotFoundError(f"❌ 파일을 찾을 수 없습니다: {aggregation_features_path}")
        
    df_agg = pl.read_parquet(aggregation_features_path)
    print(f"   ✅ 로드 성공! Shape: {df_agg.shape}")
    print(f"      - 계좌 수: {len(df_agg):,}")
    
    # -------------------------------------------------------------------------
    # 3. 그래프 피처 로드 (Graph Features)
    # -------------------------------------------------------------------------
    print(f"\n2️⃣ 그래프 피처(Add-on) 로딩 중... \n   📄 경로: {graph_features_path}")
    if not os.path.exists(graph_features_path):
        raise FileNotFoundError(f"❌ 파일을 찾을 수 없습니다: {graph_features_path}")

    df_graph = pl.read_parquet(graph_features_path)
    print(f"   ✅ 로드 성공! Shape: {df_graph.shape}")
    print(f"      - 노드 수: {len(df_graph):,}")
    
    # -------------------------------------------------------------------------
    # 4. 데이터 병합 (Left Join)
    # -------------------------------------------------------------------------
    print("\n3️⃣ 데이터 병합 (Left Join) 수행 중...")
    
    # 조인 키 통일 (account -> account_id)
    if "account" in df_graph.columns:
        df_graph = df_graph.rename({"account": "account_id"})
    
    # 타입 통일 (String)
    df_agg = df_agg.with_columns(pl.col("account_id").cast(pl.Utf8))
    df_graph = df_graph.with_columns(pl.col("account_id").cast(pl.Utf8))
    
    # 중복 컬럼 제거 (타겟값 충돌 방지)
    if "is_fraud" in df_graph.columns:
        df_graph = df_graph.drop("is_fraud")
    
    # 병합 실행
    df_integrated = df_agg.join(df_graph, on="account_id", how="left")
    print(f"   ✅ 병합 완료! Shape: {df_integrated.shape}")
    
    # -------------------------------------------------------------------------
    # 5. 결측치 처리 (Null Imputation)
    # -------------------------------------------------------------------------
    print("\n4️⃣ 결측치(Null) 처리 중 (그래프 데이터가 없는 계좌들)...")
    
    # 그래프 피처들만 골라서 0으로 채우기
    graph_cols = [c for c in df_graph.columns if c != "account_id"]
    fill_exprs = [pl.col(c).fill_null(0) for c in graph_cols]
    
    # 커뮤니티 ID는 -1로 채우기 (존재 시)
    if "community_id" in graph_cols:
        fill_exprs.append(pl.col("community_id").fill_null(-1))
        
    df_integrated = df_integrated.with_columns(fill_exprs)
    print("   ✅ 결측치 처리 완료")

    # -------------------------------------------------------------------------
    # 6. 교차 파생 피처 생성 (Interaction Features)
    # -------------------------------------------------------------------------
    print("\n5️⃣ [고급] 그래프-통계 교차 피처 생성 중...")
    
    # 필요한 컬럼이 있는지 확인 후 생성
    required_cols = {"pagerank", "sum_1h", "community_fraud_ratio", "cnt_1h", 
                     "betweenness_centrality", "burst_ratio", "total_degree", 
                     "ratio_risk_country", "clustering_coefficient", "in_out_balance_ratio"}
    
    if required_cols.issubset(set(df_integrated.columns)):
        df_integrated = df_integrated.with_columns([
            (pl.col("pagerank") * pl.col("sum_1h")).alias("pagerank_weighted_volume"),
            (pl.col("community_fraud_ratio") * pl.col("cnt_1h")).alias("community_risk_signal"),
            (pl.col("betweenness_centrality") * pl.col("burst_ratio")).alias("hub_burst_score"),
            (pl.col("total_degree") * pl.col("ratio_risk_country")).alias("network_risk_exposure"),
            (pl.col("clustering_coefficient") * pl.col("in_out_balance_ratio")).alias("internal_flow_concentration")
        ])
        print("   ✅ 교차 피처 5종 생성 완료!")
    else:
        print("   ⚠️ 일부 컬럼 부재로 교차 피처 생성 건너뜀")

    # -------------------------------------------------------------------------
    # 7. 최종 저장
    # -------------------------------------------------------------------------
    print(f"\n💾 최종 데이터 저장 중: {output_path}")
    df_integrated.write_parquet(output_path)
    
    print("\n" + "=" * 80)
    print("🎉 [성공] 모든 통합 과정이 완료되었습니다!")
    print(f"📊 최종 데이터 크기: {df_integrated.shape}")
    print(f"📁 파일 위치: {output_path}")
    print("=" * 80)

    # 데이터 미리보기
    print(df_integrated.head(3))

except Exception as e:
    print(f"\n❌ 에러 발생: {e}")
    # 상세 에러 정보 출력
    import traceback
    traceback.print_exc()

🔗 [피처 통합 프로세스] 집계 피처(V1) + 그래프 피처(V3) 병합 시작

1️⃣ 집계 피처(Base) 로딩 중... 
   📄 경로: /home/tracerofjageum/aml_features_final_advanced.parquet
   ✅ 로드 성공! Shape: (28756304, 30)
      - 계좌 수: 28,756,304

2️⃣ 그래프 피처(Add-on) 로딩 중... 
   📄 경로: ./graph_structure_only.parquet
   ✅ 로드 성공! Shape: (6, 19)
      - 노드 수: 6

3️⃣ 데이터 병합 (Left Join) 수행 중...
   ✅ 병합 완료! Shape: (28756304, 47)

4️⃣ 결측치(Null) 처리 중 (그래프 데이터가 없는 계좌들)...

❌ 에러 발생: the name 'community_id' passed to `LazyFrame.with_columns` is duplicate

It's possible that multiple expressions are returning the same default column name. If this is the case, try renaming the columns with `.alias("new_name")` to avoid duplicate column names.


Traceback (most recent call last):
  File "/tmp/ipykernel_1912/1775283355.py", line 78, in <module>
    df_integrated = df_integrated.with_columns(fill_exprs)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/dataframe/frame.py", line 10473, in with_columns
    .collect(optimizations=QueryOptFlags._eager())
     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 97, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/opt_flags.py", line 326, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/frame.py", line 2440, in collect
    return wrap_df(ldf.collect(engine,